In [1]:
import numpy as np
import pandas as pd

def bootstrap_ci(values, n_boot=10000, ci=95, agg=np.mean, seed=0):
    """
    values: 해당 config 그룹의 seed별 값들 (1D array-like)
    n_boot: 부트스트랩 리샘플링 횟수
    ci: 신뢰구간 (%)
    agg: 집계 함수 (np.mean, np.median 등)
    """
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    n = len(values)
    
    # 복원추출로 n_boot번 리샘플링 후 각각 집계
    boot_samples = rng.choice(values, size=(n_boot, n), replace=True)
    boot_stats = agg(boot_samples, axis=1)
    
    lower = np.percentile(boot_stats, (100 - ci) / 2)
    upper = np.percentile(boot_stats, 100 - (100 - ci) / 2)
    point_estimate = agg(values)
    
    return pd.Series({
        "mean": point_estimate,
        "ci_lower": lower,
        "ci_upper": upper,
        "ci_range": upper - lower,
        "count": n
    })



def bootstrap_ci_multi(df_group, cols, n_boot=10000, ci=95, agg=np.mean, seed=0):
    """
    df_group: 하나의 config 그룹에 해당하는 서브 DataFrame (여러 seed 행 포함)
    cols: 부트스트랩 계산할 컬럼 리스트
    """
    rng = np.random.default_rng(seed)
    n = len(df_group)
    
    result = {}
    for c in cols:
        values = df_group[c].to_numpy()
        boot_samples = rng.choice(values, size=(n_boot, n), replace=True)
        boot_stats = agg(boot_samples, axis=1)
        
        lower = np.percentile(boot_stats, (100 - ci) / 2)
        upper = np.percentile(boot_stats, 100 - (100 - ci) / 2)
        
        result[f"{c}_mean"] = agg(values)
        result[f"{c}_ci_lower"] = lower
        result[f"{c}_ci_upper"] = upper
        result[f"{c}_ci_range"] = upper - lower
    
    result["count"] = n
    return pd.Series(result)


In [2]:
# from $run.config $df.columns
tasks = {'walker': ['eval/task_reward_iqm','eval/flip/episode_reward_iqm', 'eval/run/episode_reward_iqm',
       'eval/walk/episode_reward_iqm', 
       'eval/stand/episode_reward_iqm'], 'quadruped':['eval/task_reward_iqm','eval/escape/episode_reward_iqm', 'eval/roll/episode_reward_iqm',
       'eval/jump/episode_reward_iqm',
       'eval/roll_fast/episode_reward_iqm', 'eval/stand/episode_reward_iqm'],
         'point_mass_maze': ['eval/task_reward_iqm','eval/reach_top_right/episode_reward_iqm',
       'eval/reach_bottom_left/episode_reward_iqm',
       'eval/reach_top_left/episode_reward_iqm',
       'eval/reach_bottom_right/episode_reward_iqm'], 'jaco': ['eval/task_reward_iqm','eval/reach_top_right/episode_reward_iqm',
       'eval/reach_bottom_left/episode_reward_iqm',
       'eval/reach_top_left/episode_reward_iqm',
       'eval/reach_bottom_right/episode_reward_iqm']}

# D-lever

In [27]:
import wandb
api = wandb.Api()
entity = api.default_entity

project = api.projects(entity)[7].name #7
runs = api.runs(f"{entity}/{project}")
project

'jepa100k'

In [28]:
# for i, proj in enumerate(api.projects(entity)):
#     print(i, proj.name)

In [29]:
configs = ['domain_name', 'tilt', 'tilt_ridge_alpha', 'tilt_temperature_start',  'tilt_temperature_end','seed', 'dataset_transitions']
configs_sep = ['domain_name', 'tilt_temperature_start', 'dataset_transitions']
criterion = 'eval/task_reward_iqm'

In [30]:
import pandas as pd

rows = []
log_interval = 20000

for i, run in enumerate(runs):
    print(i, run.state, run.name)
    if run.state == "running" or run.state == "crashed": # or run.config["algorithm"] == 'fb':
        continue
    
    df = run.history()
    ckpt = df[criterion].argmax()
    
    
    row_dict = {}

    for config in configs:
        row_dict[config] = run.config[config]
    

    for column in df.columns:
        row_dict[column] = df[column].iloc[ckpt]

    rows.append(row_dict)
    
        
#     # --- 여기부터 파일 저장 로직 ---
#     # configs 값들을 조합해서 폴더명 생성 (예: algorithm=fb_lr=0.001)
#     import os
#     folder_name = "_".join(f"{run.config[c]}" for c in configs_sep)
#     seed = run.config.get("seed", run.id)  # seed 없으면 run.id로 fallback

#     save_dir = f"./downloads/D-LEVER/{folder_name}"
#     os.makedirs(save_dir, exist_ok=True)

#     for file in run.files():
#         if file.name.endswith(".pickle"):  # 필요에 맞게 조건 조정
#             file.download(root=save_dir, replace=True)
#             downloaded_path = os.path.join(save_dir, file.name)
#             target_path = os.path.join(save_dir, f"{seed}.pt")
#             os.rename(downloaded_path, target_path)
    
#         # --- config 저장 ---
#     import json
#     config_path = os.path.join(save_dir, "config.json")
#     if not os.path.exists(config_path):  # 이미 있으면 중복 저장 방지
#         with open(config_path, "w") as f:
#             json.dump(dict(run.config), f, indent=2)

result_df = pd.DataFrame(rows)

0 finished crisp-dust-1
1 finished scarlet-water-2
2 finished helpful-oath-3
3 finished fanciful-firebrand-4
4 finished gallant-waterfall-5
5 finished sparkling-spaceship-6
6 finished cool-mountain-7
7 finished skilled-brook-8
8 finished polished-forest-9
9 finished drawn-violet-10
10 finished wild-dew-11
11 finished magic-field-12
12 finished lilac-firefly-13
13 finished fast-microwave-14
14 finished vibrant-silence-15
15 finished good-star-16
16 finished classic-paper-17
17 finished charmed-snowball-18
18 finished snowy-fog-19
19 finished eager-vortex-20
20 finished toasty-breeze-21
21 finished vague-vortex-22
22 finished lilac-waterfall-23
23 finished youthful-water-24
24 finished radiant-darkness-25
25 finished dainty-terrain-26
26 finished solar-oath-27
27 finished misty-mountain-28
28 finished sandy-oath-29
29 finished vibrant-pyramid-30
30 finished solar-universe-31
31 finished young-night-32
32 finished lyric-sponge-33
33 finished treasured-silence-34
34 finished lively-meadow-

In [31]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
configs = ['tilt', 'domain_name' , 'tilt_ridge_alpha', 'tilt_temperature_start', 'seed', 'dataset_transitions']
result_df.groupby(configs)[criterion].agg(['mean','count'])

mean  \
tilt  domain_name     tilt_ridge_alpha tilt_temperature_start seed dataset_transitions               
False jaco            0.001            20                     42   100000                40.232919   
                                                              43   100000                24.935752   
                                                              44   100000                47.529080   
                                                              45   100000                35.877831   
      point_mass_maze 0.001            20                     42   100000               104.777641   
                                                              43   100000               134.689228   
                                                              44   100000               206.108963   
      quadruped       0.001            20                     42   100000               639.530333   
                                                              43   100000               626.712637   
                                                              44   100000               610.973808   
      walker          0.001            20                     42   100000               468.904289   
                                                              43   100000               464.945656   
                                                              44   100000               497.335873   
True  jaco            0.001            10                     42   100000                50.766107   
                                                              43   100000                27.370042   
                                                              44   100000                25.132908   
                                       20                     42   100000                37.792977   
                                                              43   100000                38.329374   
                                                              44   100000                32.433504   
                      0.010            10                     42   100000                41.484579   
                                                              43   100000                53.099535   
                                                              44   100000                44.271608   
                                       20                     42   100000                14.398719   
                                                              43   100000                50.955103   
                                                              44   100000                64.066139   
                                       30                     42   100000                45.211013   
                                                              43   100000                42.353187   
                                                              44   100000                28.168315   
                      0.100            10                     42   100000                27.761279   
                                                              43   100000                37.249879   
                                                              44   100000                30.481168   
                                       20                     42   100000                26.640340   
                                                              43   100000                35.267757   
                                                              44   100000                32.941664   
      point_mass_maze 0.001            10                     42   100000               193.049431   
                                                              43   100000               250.139755   
                                                              44   100000                41.091442   
                                       20                     42   100000               129.569542   
              

In [39]:
task = 'walker'
task_df = result_df[result_df['domain_name']==task]
configs = ['domain_name', 'tilt', 'tilt_ridge_alpha', 'tilt_temperature_start',  'tilt_temperature_end', 'dataset_transitions']
result_bootstrapped = (
    task_df
    .groupby(configs)
    .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))
)

result_bootstrapped.T

/tmp/ipykernel_2435690/2066153963.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))


domain_name                                 walker                        
tilt                                          True                        
tilt_ridge_alpha                              0.01                        
tilt_temperature_start                          10          20          30
tilt_temperature_end                            10          20          30
dataset_transitions                         100000      100000      100000
eval/task_reward_iqm_mean               404.308575  421.211864  369.535390
eval/task_reward_iqm_ci_lower           378.786154  379.387721  328.104489
eval/task_reward_iqm_ci_upper           440.810108  461.765676  424.861189
eval/task_reward_iqm_ci_range            62.023954   82.377955   96.756700
eval/flip/episode_reward_iqm_mean       201.472061  299.120148  260.365231
eval/flip/episode_reward_iqm_ci_lower   122.575844  277.731583  184.818211
eval/flip/episode_reward_iqm_ci_upper   252.897865  323.705032  308.446304
eval/flip/episode_reward_iqm_ci_range   130.322021   45.973450  123.628094
eval/run/episode_reward_iqm_mean        220.824448  218.924519  197.351372
eval/run/episode_reward_iqm_ci_lower    162.884911  193.364426  154.373459
eval/run/episode_reward_iqm_ci_upper    284.646492  242.038963  275.932693
eval/run/episode_reward_iqm_ci_range    121.761581   48.674538  121.559235
eval/walk/episode_reward_iqm_mean       540.536072  541.068367  404.833191
eval/walk/episode_reward_iqm_ci_lower   379.535110  422.184486  313.877319
eval/walk/episode_reward_iqm_ci_upper   647.245514  652.847153  495.789062
eval/walk/episode_reward_iqm_ci_range   267.710403  230.662666  181.911743
eval/stand/episode_reward_iqm_mean      654.401718  625.734421  615.591766
eval/stand/episode_reward_iqm_ci_lower  550.422424  545.998260  592.987762
eval/stand/episode_reward_iqm_ci_upper  764.350952  724.489182  632.290680
eval/stand/episode_reward_iqm_ci_range  213.928528  178.490921   39.302917
count                                     4.000000    4.000000    4.000000

In [40]:
task = 'quadruped'
task_df = result_df[result_df['domain_name']==task]
configs = ['domain_name', 'tilt', 'tilt_ridge_alpha', 'tilt_temperature_start',  'tilt_temperature_end', 'dataset_transitions']
result_bootstrapped = (
    task_df
    .groupby(configs)
    .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))
)

result_bootstrapped.T

/tmp/ipykernel_2435690/1814314242.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))


domain_name                                  quadruped                        
tilt                                              True                        
tilt_ridge_alpha                                  0.01                        
tilt_temperature_start                              10          20          30
tilt_temperature_end                                10          20          30
dataset_transitions                             100000      100000      100000
eval/task_reward_iqm_mean                   561.848454  580.514289  589.159682
eval/task_reward_iqm_ci_lower               508.563891  547.231637  567.064392
eval/task_reward_iqm_ci_upper               615.133018  607.613770  604.336142
eval/task_reward_iqm_ci_range               106.569127   60.382133   37.271749
eval/escape/episode_reward_iqm_mean          56.834466   67.300874   68.230309
eval/escape/episode_reward_iqm_ci_lower      48.254766   56.490787   46.927518
eval/escape/episode_reward_iqm_ci_upper      65.414165   77.209671  105.395665
eval/escape/episode_reward_iqm_ci_range      17.159399   20.718884   58.468147
eval/roll/episode_reward_iqm_mean           786.795166  822.834335  874.177917
eval/roll/episode_reward_iqm_ci_lower       715.920288  759.717941  865.430237
eval/roll/episode_reward_iqm_ci_upper       855.317566  883.826355  884.310974
eval/roll/episode_reward_iqm_ci_range       139.397278  124.108414   18.880737
eval/jump/episode_reward_iqm_mean           654.610886  646.178009  678.803650
eval/jump/episode_reward_iqm_ci_lower       580.376266  593.846008  645.382935
eval/jump/episode_reward_iqm_ci_upper       712.628571  698.510010  702.138062
eval/jump/episode_reward_iqm_ci_range       132.252304  104.664001   56.755127
eval/roll_fast/episode_reward_iqm_mean      487.182648  492.437775  474.340744
eval/roll_fast/episode_reward_iqm_ci_lower  432.902596  469.427086  417.828438
eval/roll_fast/episode_reward_iqm_ci_upper  520.147644  518.864922  525.562927
eval/roll_fast/episode_reward_iqm_ci_range   87.245049   49.437836  107.734489
eval/stand/episode_reward_iqm_mean          823.819107  873.820450  850.245789
eval/stand/episode_reward_iqm_ci_lower      707.728745  809.713837  792.375427
eval/stand/episode_reward_iqm_ci_upper      929.788513  916.733856  894.856689
eval/stand/episode_reward_iqm_ci_range      222.059769  107.020020  102.481262
count                                         4.000000    4.000000    4.000000

In [41]:
task = 'jaco'
task_df = result_df[result_df['domain_name']==task]
configs = ['domain_name', 'tilt', 'tilt_ridge_alpha', 'tilt_temperature_start',  'tilt_temperature_end', 'dataset_transitions']
result_bootstrapped = (
    task_df
    .groupby(configs)
    .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))
)

result_bootstrapped.T

/tmp/ipykernel_2435690/4026918772.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))


domain_name                                              jaco             \
tilt                                                     True              
tilt_ridge_alpha                                         0.01              
tilt_temperature_start                                     10         20   
tilt_temperature_end                                       10         20   
dataset_transitions                                    100000     100000   
eval/task_reward_iqm_mean                           15.390530  14.032355   
eval/task_reward_iqm_ci_lower                        9.198386  10.828943   
eval/task_reward_iqm_ci_upper                       20.314935  17.235768   
eval/task_reward_iqm_ci_range                       11.116548   6.406825   
eval/reach_top_right/episode_reward_iqm_mean        32.329208  35.360373   
eval/reach_top_right/episode_reward_iqm_ci_lower    12.061096  31.027376   
eval/reach_top_right/episode_reward_iqm_ci_upper    46.750072  39.693371   
eval/reach_top_right/episode_reward_iqm_ci_range    34.688976   8.665995   
eval/reach_bottom_left/episode_reward_iqm_mean       3.602076   7.248397   
eval/reach_bottom_left/episode_reward_iqm_ci_lower   0.738527   0.397925   
eval/reach_bottom_left/episode_reward_iqm_ci_upper   9.176194  14.098869   
eval/reach_bottom_left/episode_reward_iqm_ci_range   8.437668  13.700945   
eval/reach_top_left/episode_reward_iqm_mean         18.210473   1.054354   
eval/reach_top_left/episode_reward_iqm_ci_lower      4.187171   0.771579   
eval/reach_top_left/episode_reward_iqm_ci_upper     32.623997   1.337130   
eval/reach_top_left/episode_reward_iqm_ci_range     28.436826   0.565551   
eval/reach_bottom_right/episode_reward_iqm_mean      7.420362  12.466297   
eval/reach_bottom_right/episode_reward_iqm_ci_l...   1.171244  10.553342   
eval/reach_bottom_right/episode_reward_iqm_ci_u...  11.369083  14.379252   
eval/reach_bottom_right/episode_reward_iqm_ci_r...  10.197839   3.825911   
count                                                3.000000   2.000000   

domain_name                                                    
tilt                                                           
tilt_ridge_alpha                                               
tilt_temperature_start                                     30  
tilt_temperature_end                                       30  
dataset_transitions                                    100000  
eval/task_reward_iqm_mean                           14.859136  
eval/task_reward_iqm_ci_lower                       12.812393  
eval/task_reward_iqm_ci_upper                       16.905879  
eval/task_reward_iqm_ci_range                        4.093486  
eval/reach_top_right/episode_reward_iqm_mean        17.629815  
eval/reach_top_right/episode_reward_iqm_ci_lower     8.450931  
eval/reach_top_right/episode_reward_iqm_ci_upper    26.808699  
eval/reach_top_right/episode_reward_iqm_ci_range    18.357768  
eval/reach_bottom_left/episode_reward_iqm_mean      15.035460  
eval/reach_bottom_left/episode_reward_iqm_ci_lower  13.666142  
eval/reach_bottom_left/episode_reward_iqm_ci_upper  16.404778  
eval/reach_bottom_left/episode_reward_iqm_ci_range   2.738635  
eval/reach_top_left/episode_reward_iqm_mean          2.400583  
eval/reach_top_left/episode_reward_iqm_ci_lower      1.575718  
eval/reach_top_left/episode_reward_iqm_ci_upper      3.225449  
eval/reach_top_left/episode_reward_iqm_ci_range      1.649730  
eval/reach_bottom_right/episode_reward_iqm_mean     24.370687  
eval/reach_bottom_right/episode_reward_iqm_ci_l...  23.923227  
eval/reach_bottom_right/episode_reward_iqm_ci_u...  24.818146  
eval/reach_bottom_right/episode_reward_iqm_ci_r...   0.894918  
count                                                2.000000

In [42]:
task = 'point_mass_maze'
task_df = result_df[result_df['domain_name']==task]
configs = ['domain_name', 'tilt', 'tilt_ridge_alpha', 'tilt_temperature_start',  'tilt_temperature_end', 'dataset_transitions']

result_bootstrapped = (
    task_df
    .groupby(configs)
    .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))
)

result_bootstrapped.T

/tmp/ipykernel_2435690/805778914.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: bootstrap_ci_multi(g, tasks[task], n_boot=10000, ci=95))


domain_name                                        point_mass_maze  \
tilt                                                          True   
tilt_ridge_alpha                                              0.01   
tilt_temperature_start                                          10   
tilt_temperature_end                                            10   
dataset_transitions                                         100000   
eval/task_reward_iqm_mean                               282.551031   
eval/task_reward_iqm_ci_lower                           244.851439   
eval/task_reward_iqm_ci_upper                           338.570549   
eval/task_reward_iqm_ci_range                            93.719110   
eval/reach_top_right/episode_reward_iqm_mean            357.751307   
eval/reach_top_right/episode_reward_iqm_ci_lower          3.451191   
eval/reach_top_right/episode_reward_iqm_ci_upper        712.051422   
eval/reach_top_right/episode_reward_iqm_ci_range        708.600231   
eval/reach_bottom_left/episode_reward_iqm_mean           59.856709   
eval/reach_bottom_left/episode_reward_iqm_ci_lower        0.000000   
eval/reach_bottom_left/episode_reward_iqm_ci_upper      179.570126   
eval/reach_bottom_left/episode_reward_iqm_ci_range      179.570126   
eval/reach_top_left/episode_reward_iqm_mean             712.596107   
eval/reach_top_left/episode_reward_iqm_ci_lower         409.698845   
eval/reach_top_left/episode_reward_iqm_ci_upper         909.590332   
eval/reach_top_left/episode_reward_iqm_ci_range         499.891487   
eval/reach_bottom_right/episode_reward_iqm_mean           0.000000   
eval/reach_bottom_right/episode_reward_iqm_ci_l...        0.000000   
eval/reach_bottom_right/episode_reward_iqm_ci_u...        0.000000   
eval/reach_bottom_right/episode_reward_iqm_ci_r...        0.000000   
count                                                     4.000000   

domain_name                                                                 
tilt                                                                        
tilt_ridge_alpha                                                            
tilt_temperature_start                                      20          30  
tilt_temperature_end                                        20          30  
dataset_transitions                                     100000      100000  
eval/task_reward_iqm_mean                           273.755492  320.204291  
eval/task_reward_iqm_ci_lower                       207.667162  262.924278  
eval/task_reward_iqm_ci_upper                       373.528538  377.484303  
eval/task_reward_iqm_ci_range                       165.861375  114.560024  
eval/reach_top_right/episode_reward_iqm_mean        182.665329  177.413895  
eval/reach_top_right/episode_reward_iqm_ci_lower      0.000000    0.000000  
eval/reach_top_right/episode_reward_iqm_ci_upper    547.995987  532.241684  
eval/reach_top_right/episode_reward_iqm_ci_range    547.995987  532.241684  
eval/reach_bottom_left/episode_reward_iqm_mean        6.834390  446.788773  
eval/reach_bottom_left/episode_reward_iqm_ci_lower    0.000000  168.660004  
eval/reach_bottom_left/episode_reward_iqm_ci_upper   20.503170  724.917542  
eval/reach_bottom_left/episode_reward_iqm_ci_range   20.503170  556.257538  
eval/reach_top_left/episode_reward_iqm_mean         905.522247  656.614495  
eval/reach_top_left/episode_reward_iqm_ci_lower     823.954681  224.304300  
eval/reach_top_left/episode_reward_iqm_ci_upper     947.471741  912.837738  
eval/reach_top_left/episode_reward_iqm_ci_range     123.517059  688.533438  
eval/reach_bottom_right/episode_reward_iqm_mean       0.000000    0.000000  
eval/reach_bottom_right/episode_reward_iqm_ci_l...    0.000000    0.000000  
eval/reach_bottom_right/episode_reward_iqm_ci_u...    0.000000    0.000000  
eval/reach_bottom_right/episode_reward_iqm_ci_r...    0.000000    0.000000  
count                                                 4.000000    4.000000

# D-LEVER

In [28]:
configs = ['domain_name', 'tilt', 'tilt_ridge_alpha', 
           'tilt_temperature_start', 'tilt_temperature_end', 
           'dataset_transitions', 'seed']
col = 'eval/task_reward_iqm'

# seed는 groupby 기준에서 빼고, 그룹 내에서 seed별 값들을 모아서 부트스트랩
group_cols = ['domain_name', 'tilt', 'tilt_ridge_alpha', 
              'tilt_temperature_start', 'tilt_temperature_end', 
              'dataset_transitions']

result_bootstrapped = (
    result_df
    .groupby(group_cols)[col]
    .apply(lambda x: bootstrap_ci(x, n_boot=10000, ci=95))
    .unstack()
)

result_bootstrapped

mean  \
domain_name     tilt tilt_ridge_alpha tilt_temperature_start tilt_temperature_end dataset_transitions               
jaco            True 0.01             10                     10                   100000                16.658269   
                                      20                     20                   100000                10.828943   
                                      30                     30                   100000                16.905879   
point_mass_maze True 0.01             10                     10                   100000               300.870956   
                                      20                     20                   100000               299.717390   
                                      30                     30                   100000               337.185965   
quadruped       True 0.01             10                     10                   100000               561.848454   
                                      20                     20                   100000               580.514289   
                                      30                     30                   100000               584.961288   
walker          True 0.01             10                     10                   100000               404.308575   
                                      20                     20                   100000               421.211864   
                                      30                     30                   100000               369.535390   

                                                                                                         ci_lower  \
domain_name     tilt tilt_ridge_alpha tilt_temperature_start tilt_temperature_end dataset_transitions               
jaco            True 0.01             10                     10                   100000                16.658269   
                                      20                     20                   100000                10.828943   
                                      30                     30                   100000                16.905879   
point_mass_maze True 0.01             10                     10                   100000               237.019363   
                                      20                     20                   100000               237.082291   
                                      30                     30                   100000               256.589291   
quadruped       True 0.01             10                     10                   100000               508.563891   
                                      20                     20                   100000               547.231637   
                                      30                     30                   100000               555.500902   
walker          True 0.01             10                     10                   100000               378.786154   
                                      20                     20                   100000               379.387721   
                                      30                     30                   100000               328.104489   

                                                                                                         ci_upper  \
domain_name     tilt tilt_ridge_alpha tilt_temperature_start tilt_temperature_end dataset_transitions               
jaco            True 0.01             10                     10                   100000                16.658269   
                                      20                     20                   100000                10.828943   
                                      30                     30                   100000                16.905879   
point_mass_maze True 0.01             10                     10                   100000               364.722549   
                                      20                     20                   100000               419.010620   
            

# FB

In [12]:
#

,,,,,mean,sem,count
domain_name,tilt,tilt_ridge_alpha,tilt_temperature_start,dataset_transitions,,,
jaco,False,0.1,20,100000,16.074499,4.484291,3
point_mass_maze,False,0.1,20,100000,234.504674,1.560986,3
quadruped,False,0.1,20,100000,586.628774,28.297979,4
walker,False,0.1,20,100000,393.016110,11.093169,5


In [ ]:
# df_group_by = result_df.groupby(configs)[col].agg(["mean",  "sem"]).to_csv("result.csv")

In [58]:
# columns = ['eval/task_reward_iqm','eval/reach_top_left/episode_reward_iqm',
# 'eval/reach_top_right/episode_reward_iqm',
# 'eval/reach_bottom_left/episode_reward_iqm',
# 'eval/reach_bottom_right/episode_reward_iqm',
#            ]